In [ ]:
!pip install -q torch torchvision matplotlib


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 47.6 MB/s eta 0:00:00


In [ ]:
# GAN Implementation for Mango Leaf Disease Image Generation
# Import necessary libraries
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt
from google.colab import drive
import time
from glob import glob
from skimage.metrics import structural_similarity as ssim
from tensorflow.keras.applications.inception_v3 import InceptionV3
from tensorflow.keras.applications.inception_v3 import preprocess_input
import numpy as np
from scipy import linalg
import cv2
from tqdm import tqdm


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import zipfile
import os

# Define path to your zip file in Google Drive
zip_path = '/content/drive/MyDrive/Mango-Leaf-Disease_dataset.zip'  # Update path if needed

# Extract to a known location
extract_path = '/content/mango_dataset'
os.makedirs(extract_path, exist_ok=True)

# Unzip
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset unzipped successfully.")


Dataset unzipped successfully.


# Data Preprocessing & Loader Setup

In [ ]:
!pip uninstall sympy -y
!pip install sympy --upgrade


Found existing installation: sympy 1.13.1
Uninstalling sympy-1.13.1:
  Successfully uninstalled sympy-1.13.1
  Using cached sympy-1.13.3-py3-none-any.whl.metadata (12 kB)
Using cached sympy-1.13.3-py3-none-any.whl (6.2 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires sympy==1.13.1; python_version >= "3.9", but you have sympy 1.13.3 which is incompatible.


In [ ]:
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

# Define transform: resize to 64x64 and normalize to [-1, 1]
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

# Create dataset and dataloader
data_path = '/content/mango_dataset'
dataset = ImageFolder(data_path, transform=transform)
dataloader = DataLoader(dataset, batch_size=128, shuffle=True, num_workers=0)


# Print class mapping
print("Class-to-index mapping:", dataset.class_to_idx)
n_classes = len(dataset.classes)

n_classes = len(dataset.classes)  # This line must be defined before model creation
print("Number of classes:", n_classes)


Class-to-index mapping: {'Mango-Leaf-Disease_dataset': 0}
Number of classes: 1


# Define Generator & Discriminator (Conditional DCGAN)

In [ ]:
import torch
import torch.nn as nn

# Parameters
nz = 100      # Latent vector size
ngf = 64      # Generator filters
ndf = 64      # Discriminator filters
nc = 3        # RGB images
embed_size = 50
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Generator
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.label_emb = nn.Embedding(n_classes, embed_size)
        self.net = nn.Sequential(
            nn.ConvTranspose2d(nz + embed_size, ngf*8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf*8), nn.ReLU(True),
            nn.ConvTranspose2d(ngf*8, ngf*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf*4), nn.ReLU(True),
            nn.ConvTranspose2d(ngf*4, ngf*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf*2), nn.ReLU(True),
            nn.ConvTranspose2d(ngf*2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf), nn.ReLU(True),
            nn.ConvTranspose2d(ngf, nc, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, noise, labels):
        label_input = self.label_emb(labels).unsqueeze(2).unsqueeze(3)
        x = torch.cat([noise, label_input], 1)
        return self.net(x)

# Discriminator
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.label_emb = nn.Embedding(n_classes, 1)
        self.net = nn.Sequential(
            nn.Conv2d(nc + 1, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf, ndf*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*2), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*2, ndf*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*4), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf*4, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, img, labels):
        label_input = self.label_emb(labels).unsqueeze(2).unsqueeze(3).expand(-1, -1, 64, 64)
        x = torch.cat([img, label_input], 1)
        return self.net(x)


# Training the cDCGAN

In [ ]:
# Initialize models
netG = Generator().to(device)
netD = Discriminator().to(device)

# Loss and optimizers
criterion = nn.BCELoss()
optimizerD = torch.optim.Adam(netD.parameters(), lr=0.0002, betas=(0.5, 0.999))
optimizerG = torch.optim.Adam(netG.parameters(), lr=0.0002, betas=(0.5, 0.999))

# Training settings
epochs = 50
fixed_noise = torch.randn(30, nz, 1, 1, device=device)
real_label, fake_label = 1.0, 0.0

print("Starting training...")

for epoch in range(1, epochs + 1):
    for i, (real_imgs, labels) in enumerate(dataloader):
        b_size = real_imgs.size(0)
        real_imgs = real_imgs.to(device)
        labels = labels.to(device)

        ### Train Discriminator ###
        netD.zero_grad()
        output_real = netD(real_imgs, labels).view(-1)
        loss_real = criterion(output_real, torch.full_like(output_real, real_label, device=device))

        noise = torch.randn(b_size, nz, 1, 1, device=device)
        fake_imgs = netG(noise, labels)
        output_fake = netD(fake_imgs.detach(), labels).view(-1)
        loss_fake = criterion(output_fake, torch.full_like(output_fake, fake_label, device=device))

        d_loss = loss_real + loss_fake
        d_loss.backward()
        optimizerD.step()

        ### Train Generator ###
        netG.zero_grad()
        output = netD(fake_imgs, labels).view(-1)
        g_loss = criterion(output, torch.full_like(output, real_label, device=device))
        g_loss.backward()
        optimizerG.step()

    print(f"Epoch [{epoch}/{epochs}] - Loss_D: {d_loss.item():.4f}, Loss_G: {g_loss.item():.4f}")

# ✅ Save models to Google Drive
torch.save(netG.state_dict(), "/content/drive/MyDrive/cDCGAN_generator.pth")
torch.save(netD.state_dict(), "/content/drive/MyDrive/cDCGAN_discriminator.pth")
print("✅ Models saved to Google Drive.")


Starting training...
Epoch [1/50] - Loss_D: 1.1270, Loss_G: 1.2319
Epoch [2/50] - Loss_D: 0.6755, Loss_G: 1.7359
Epoch [3/50] - Loss_D: 0.8220, Loss_G: 2.1384
Epoch [4/50] - Loss_D: 1.1739, Loss_G: 1.6757
Epoch [5/50] - Loss_D: 0.9640, Loss_G: 1.8144
Epoch [6/50] - Loss_D: 0.9143, Loss_G: 1.8327
Epoch [7/50] - Loss_D: 0.8159, Loss_G: 2.0040
Epoch [8/50] - Loss_D: 0.7247, Loss_G: 1.6820
Epoch [9/50] - Loss_D: 0.9180, Loss_G: 2.7706
Epoch [10/50] - Loss_D: 0.5569, Loss_G: 2.0998
Epoch [11/50] - Loss_D: 0.7898, Loss_G: 1.3730
Epoch [12/50] - Loss_D: 0.6835, Loss_G: 1.4748
Epoch [13/50] - Loss_D: 1.3876, Loss_G: 0.6268
Epoch [14/50] - Loss_D: 0.6573, Loss_G: 2.0805
Epoch [15/50] - Loss_D: 0.6262, Loss_G: 2.4556
Epoch [16/50] - Loss_D: 0.9952, Loss_G: 3.5090
Epoch [17/50] - Loss_D: 1.0043, Loss_G: 1.7566
Epoch [18/50] - Loss_D: 0.9396, Loss_G: 3.4206
Epoch [19/50] - Loss_D: 0.9050, Loss_G: 1.2644
Epoch [20/50] - Loss_D: 0.7246, Loss_G: 1.9859
Epoch [21/50] - Loss_D: 0.5765, Loss_G: 2.7746
E

In [ ]:
import torchvision.utils as vutils
import os

# Load trained generator
netG.load_state_dict(torch.load("/content/drive/MyDrive/cDCGAN_generator.pth", map_location=device))
netG.eval()

# Save in Drive
output_dir = "/content/drive/MyDrive/generated_images"
os.makedirs(output_dir, exist_ok=True)

# Generate 30 images for each class
for class_idx in range(n_classes):
    noise = torch.randn(30, nz, 1, 1, device=device)
    labels = torch.full((30,), class_idx, dtype=torch.long, device=device)

    with torch.no_grad():
        fake_imgs = netG(noise, labels).detach().cpu()

    class_name = dataset.classes[class_idx].replace(" ", "_")
    class_dir = os.path.join(output_dir, class_name)
    os.makedirs(class_dir, exist_ok=True)

    for i, img in enumerate(fake_imgs):
        vutils.save_image(img, f"{class_dir}/{class_name}_{i+1:03d}.png", normalize=True)

print("✅ Images saved in: /content/drive/MyDrive/generated_images")


✅ Images saved in: /content/drive/MyDrive/generated_images
